In [ ]:
# Cell 1: Install and upgrade packages
!pip install -q --upgrade pip setuptools wheel
!pip install -q -r requirements.txt

In [ ]:
# Cell 2: Imports and SageMaker Configuration
import os
import boto3
import pandas as pd
import numpy as np

# Set environment variables to handle connection issues
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

# Import SageMaker components correctly for 3.x
try:
    from sagemaker.session import Session
    from sagemaker import get_execution_role
    sagemaker_available = True
except ImportError:
    try:
        from sagemaker import Session, get_execution_role
        sagemaker_available = True
    except ImportError:
        print("SageMaker imports not available - using fallback configuration")
        Session = None
        get_execution_role = None
        sagemaker_available = False

# SageMaker session and role setup
region = "us-east-1"  # Default region
role = "arn:aws:iam::123456789012:role/service-role/AmazonSageMaker-ExecutionRole"  # Placeholder
sess = None

if sagemaker_available and Session:
    try:
        sess = Session()
        region = sess.boto_region_name
        if get_execution_role:
            role = get_execution_role()
    except Exception as e:
        print(f"Using fallback configuration due to: {e}")
        sess = None

bucket = "sagemaker-hyd-house-rajini-2026"
prefix = "hyd-house-price"

print(f"Environment ready in {region}")
print(f"Using Role: {role}")

In [ ]:
# Cell 3: Load and Clean Dataset
def load_dataset_safely():
    try:
        from datasets import load_dataset
        print("Loading dataset from Hugging Face...")
        ds = load_dataset("Saathwik56/houseprice")
        df = ds["train"].to_pandas()
        print(f"Dataset loaded successfully with shape: {df.shape}")
        return df
    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("Creating sample house price data for testing...")
        np.random.seed(42)
        n_samples = 1000
        sample_data = {
            'bedrooms': np.random.randint(1, 6, n_samples),
            'bathrooms': np.random.randint(1, 4, n_samples),
            'sqft_living': np.random.randint(500, 5000, n_samples),
            'sqft_lot': np.random.randint(1000, 10000, n_samples),
            'floors': np.random.randint(1, 4, n_samples),
            'waterfront': np.random.choice([0, 1], n_samples, p=[0.9, 0.1]),
            'view': np.random.randint(0, 5, n_samples),
            'condition': np.random.randint(1, 6, n_samples),
            'grade': np.random.randint(3, 13, n_samples),
            'yr_built': np.random.randint(1900, 2020, n_samples),
            'yr_renovated': np.random.choice([0] + list(range(1950, 2020)), n_samples, p=[0.7] + [0.3/70]*70),
            'price': np.random.randint(100000, 2000000, n_samples)
        }
        df = pd.DataFrame(sample_data)
        print(f"Sample dataset created with shape: {df.shape}")
        return df

# Load the dataset
df = load_dataset_safely()

# Clean column names (lower case, remove spaces)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

print(f"Dataset loaded with shape: {df.shape}")
print("Columns:", list(df.columns))
df.head()

In [ ]:
# Cell 4: Data Cleaning and Splitting
from sklearn.model_selection import train_test_split

# Identify target column
target = "price" if "price" in df.columns else df.columns[-1]
print(f"Target column: {target}")

# Drop rows where target is missing
df = df.dropna(subset=[target])
print(f"After dropping missing target values: {df.shape}")

# Clean numeric columns that are currently strings (like "1,200,000")
for c in df.columns:
    if df[c].dtype == "object":
        # Remove commas and attempt conversion to numeric
        temp_col = df[c].astype(str).str.replace(",", "", regex=False)
        df[c] = pd.to_numeric(temp_col, errors="ignore")

# Split the data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Create local directory and save files
os.makedirs("data", exist_ok=True)
train_df.to_csv("data/train.csv", index=False)
test_df.to_csv("data/test.csv", index=False)

print(f"Files saved locally: data/train.csv ({len(train_df)} rows), data/test.csv ({len(test_df)} rows)")
print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

In [ ]:
# Cell 5: Upload to S3 (if SageMaker session available)
if sess:
    try:
        train_path = sess.upload_data(path="data/train.csv", bucket=bucket, key_prefix=f"{prefix}/train")
        test_path = sess.upload_data(path="data/test.csv", bucket=bucket, key_prefix=f"{prefix}/test")
        print(f"Train data uploaded to: {train_path}")
        print(f"Test data uploaded to: {test_path}")
    except Exception as e:
        print(f"Error uploading to S3: {e}")
        print("Continuing with local files...")
        train_path = "data/train.csv"
        test_path = "data/test.csv"
else:
    print("No SageMaker session available - using local files")
    train_path = "data/train.csv"
    test_path = "data/test.csv"

In [ ]:
# Cell 6: Create Training Script
training_script = '''
import pandas as pd
import os
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

if __name__ == "__main__":
    # Handle SageMaker Paths
    train_dir = os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train")
    model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
    
    # For local testing
    if not os.path.exists(train_dir):
        train_dir = "data"
    if not os.path.exists(model_dir):
        model_dir = "model"
        os.makedirs(model_dir, exist_ok=True)

    # Load Data
    train_path = os.path.join(train_dir, "train.csv")
    df = pd.read_csv(train_path)
    print(f"Loaded training data with shape: {df.shape}")

    # Identify Target and Features
    target = "price" if "price" in df.columns else df.columns[-1]
    
    # Select only numeric columns for features
    X = df.drop(columns=[target]).select_dtypes(include=['number'])
    y = df[target]
    
    print(f"Training with {len(X.columns)} features: {list(X.columns)}")
    print(f"Target: {target}")

    # Train Model
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X, y)
    
    # Calculate training metrics
    y_pred = model.predict(X)
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    
    print(f"Training MSE: {mse:.2f}")
    print(f"Training R2: {r2:.4f}")

    # Save Model
    model_path = os.path.join(model_dir, "model.joblib")
    joblib.dump(model, model_path)
    print(f"Model saved to: {model_path}")

# Inference function for SageMaker deployment
def model_fn(model_dir):
    """Load the model from the model_dir"""
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    return model
'''

# Write the training script to file
with open("train.py", "w") as f:
    f.write(training_script)

print("Training script created: train.py")

In [ ]:
# Cell 7: Test Training Script Locally
print("Testing training script locally...")
!python train.py

In [ ]:
# Cell 8: Load and Test the Trained Model
import joblib
from sklearn.metrics import mean_squared_error, r2_score

# Load the trained model
model = joblib.load("model/model.joblib")
print("Model loaded successfully")

# Test on test data
test_data = pd.read_csv("data/test.csv")
target = "price" if "price" in test_data.columns else test_data.columns[-1]
X_test = test_data.drop(columns=[target]).select_dtypes(include=['number'])
y_test = test_data[target]

# Make predictions
y_pred = model.predict(X_test)

# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"Test Results:")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

# Show some predictions
print("\nSample Predictions:")
comparison = pd.DataFrame({
    'Actual': y_test.iloc[:10].values,
    'Predicted': y_pred[:10],
    'Difference': y_test.iloc[:10].values - y_pred[:10]
})
print(comparison)

In [ ]:
# Cell 9: Cleanup (Optional)
import shutil

print("Lab completed successfully!")
print("\nFiles created:")
print("- data/train.csv")
print("- data/test.csv")
print("- train.py")
print("- model/model.joblib")

# Uncomment the following lines if you want to clean up files
# if os.path.exists("data"):
#     shutil.rmtree("data")
#     print("Removed data directory")
# if os.path.exists("model"):
#     shutil.rmtree("model")
#     print("Removed model directory")
# if os.path.exists("train.py"):
#     os.remove("train.py")
#     print("Removed train.py")